In [14]:
import pandas as pd
import requests
import gower
import pandas as pd

In [15]:
url = "https://raw.githubusercontent.com/propublica/compas-analysis/refs/heads/master/compas-scores-raw.csv"

In [16]:
response = requests.get(url)
if response.status_code == 200:
    with open("downloaded_file.csv", "wb") as file:
        file.write(response.content)
    compas_initial = pd.read_csv("downloaded_file.csv")
    print("Data loaded into DataFrame:")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

Data loaded into DataFrame:


In [17]:
seen_ids = set()
for index, row in compas_initial.iterrows():
    if row['Person_ID'] not in seen_ids:
        seen_ids.add(row['Person_ID'])
    else:
        compas_initial.drop(index=index, inplace=True)

In [5]:
compas_initial.shape

(18610, 28)

In [6]:
compas_initial

,Person_ID,AssessmentID,Case_ID,Agency_Text,LastName,FirstName,MiddleName,Sex_Code_Text,Ethnic_Code_Text,DateOfBirth,...,RecSupervisionLevel,RecSupervisionLevelText,Scale_ID,DisplayText,RawScore,DecileScore,ScoreText,AssessmentType,IsCompleted,IsDeleted
0,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,...,1,Low,7,Risk of Violence,-2.08,4,Low,New,1,0
1,50848,57174,51956,PRETRIAL,KENDALL,KEVIN,NaN,Male,Caucasian,09/16/84,...,1,Low,7,Risk of Violence,-2.84,2,Low,New,1,0
2,50855,57181,51963,PRETRIAL,DAYES,DANIEL,NaN,Male,African-American,08/25/94,...,4,High,7,Risk of Violence,-1.20,8,High,New,1,0
3,50850,57176,51958,PRETRIAL,Debe,Mikerlie,George,Female,African-American,10/09/94,...,2,Medium,7,Risk of Violence,-1.29,7,Medium,New,1,0
4,50839,57162,51945,PRETRIAL,McLaurin,Stephanie,Nicole,Female,African-American,06/29/85,...,1,Low,7,Risk of Violence,-2.90,2,Low,New,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18605,19968,79664,36500,PRETRIAL,BUTTERFIELD,JAMES,MICHAEL,Male,Caucasian,09/24/73,...,2,Medium,7,Risk of Violence,-2.44,3,Low,Copy,1,0
18606,68608,79677,72045,PRETRIAL,Hayes,Thomas,NaN,Male,Caucasian,08/13/89,...,1,Low,7,Risk of Violence,-1.74,5,Medium,New,1,0
18607,68595,79656,72031,PRETRIAL,PENA,ROLANDO,N,Male,Hispanic,06/13/85,...,2,Medium,7,Risk of Violence,-1.61,6,Medium,New,1,0
18608,68598,79660,72035,PRETRIAL,SUAREZ,ANDERSON,NaN,Male,Caucasian,08/10/81,...,1,Low,7,Risk of Violence,-3.12,1,Low,New,1,0


In [7]:
compas_filtered_columns = compas_initial.drop(['AssessmentType','Screening_Date','AssessmentID','Case_ID','Person_ID','LastName','FirstName','MiddleName','ScaleSet_ID','ScaleSet','AssessmentReason','RecSupervisionLevel','IsCompleted','IsDeleted'],axis=1)	

In [8]:
compas_filtered_columns.columns

Index(['Agency_Text', 'Sex_Code_Text', 'Ethnic_Code_Text', 'DateOfBirth',
       'Language', 'LegalStatus', 'CustodyStatus', 'MaritalStatus',
       'RecSupervisionLevelText', 'Scale_ID', 'DisplayText', 'RawScore',
       'DecileScore', 'ScoreText'],
      dtype='object')

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
def reduce_to2(x):
    if x != 'Caucasian':
        return 'No Caucasian'
    else:
        return x
compas_filtered_columns['C_Agency_Text'] = le.fit_transform( compas_filtered_columns['Agency_Text'] )
compas_filtered_columns['C_Sex_Code_Text'] = le.fit_transform( compas_filtered_columns['Sex_Code_Text'] )
compas_filtered_columns['C_Ethnic_Code_Text'] = compas_filtered_columns['Ethnic_Code_Text'].apply(lambda x : reduce_to2(x))
compas_filtered_columns['C_LegalStatus'] = le.fit_transform( compas_filtered_columns['LegalStatus'] )
compas_filtered_columns['C_Language'] = le.fit_transform( compas_filtered_columns['Language'] )
compas_filtered_columns['C_CustodyStatus'] = le.fit_transform( compas_filtered_columns['CustodyStatus'] )
compas_filtered_columns['C_MaritalStatus'] = le.fit_transform( compas_filtered_columns['MaritalStatus'] )
compas_filtered_columns['C_RecSupervisionLevelText'] = le.fit_transform( compas_filtered_columns['RecSupervisionLevelText'] )
compas_filtered_columns['C_DisplayText'] = le.fit_transform( compas_filtered_columns['DisplayText'] )
compas_filtered_columns['C_ScoreText'] = le.fit_transform( compas_filtered_columns['ScoreText'] )
compas_filtered_columns['YearOfBirth'] = compas_filtered_columns['DateOfBirth'].apply(lambda x : x.split('/')[-1])

compas_to_analysis = compas_filtered_columns.drop(['Agency_Text', 'Sex_Code_Text' ,'Ethnic_Code_Text','LegalStatus', 'Language','CustodyStatus' ,'MaritalStatus' ,'RecSupervisionLevelText' ,'DisplayText', 'ScoreText','DateOfBirth'],axis=1)
compas_to_analysis['YearOfBirth'] = compas_to_analysis['YearOfBirth'].astype('int64')

In [10]:
from kafkanator.dataviz import similar_subjects_treatment_plot
from kafkanator.dataviz import similar_subjects_dashboard
from kafkanator.fairness.simmilarity import simmilarity_fairness_hash

In [11]:
tosim = compas_to_analysis[['Scale_ID',	'RawScore'	,'DecileScore'	,'C_Agency_Text',	'C_Sex_Code_Text',	'C_Ethnic_Code_Text',	'C_LegalStatus'	,'C_Language',	'C_CustodyStatus',	'C_MaritalStatus',	'C_RecSupervisionLevelText',	'C_DisplayText',	'YearOfBirth']]

In [12]:
appli = similar_subjects_dashboard(tosim , 'C_Ethnic_Code_Text' ,['Caucasian','No Caucasian'] ,  500 , compas_to_analysis['C_ScoreText' ]   )

 wo target  0      1
1      1
6      1
9      1
11     1
      ..
491    1
494    1
495    2
496    1
498    2
Name: C_ScoreText, Length: 168, dtype: int64  dtype  int64
 ma target  2      0
3      2
4      1
5      2
7      2
      ..
488    1
492    2
493    0
497    1
499    1
Name: C_ScoreText, Length: 332, dtype: int64  dtype  int64
getting distance matrix, data shape 
finish distance matrix
x data  [347, 347, 285, 285, 347, 362, 457, 0, 0, 498]  y data  [405, 37, 240, 371, 120, 223, 337, 371, 166, 275] z data  [0.08682855, 0.087168165, 0.08746723, 0.08746723, 0.08746723, 0.08780685, 0.0878474, 0.08865589, 0.088825695, 0.08886625]
returning fig


In [13]:
if __name__ == '__main__':
    appli.run(debug=True, port=8050)